# 01 · SQL analysis
Reads the tables and views built by `python src/run_sql.py` (which runs `sql/01_schema.sql` … `sql/08_views.sql` against `data/grid.duckdb`).

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))
import duckdb, numpy as np, pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
con = duckdb.connect("data/grid.duckdb", read_only=True)
con.execute("SET TimeZone = 'UTC'")
q = lambda sql: con.execute(sql).df()

## Row counts

In [ ]:
tables = ["raw_demand", "raw_generation", "raw_weather", "dim_fuel", "dim_calendar",
          "fact_load", "fact_generation", "fact_weather", "model_hourly"]
pd.DataFrame({"table": tables, "rows": [q(f"SELECT COUNT(*) n FROM {t}").n[0] for t in tables]})

## Data quality (sql/02_quality.sql)

In [ ]:
q("SELECT * FROM qa_summary")

In [ ]:
q("SELECT * FROM qa_missing_demand")

In [ ]:
q("""SELECT CAST(local_ts AS DATE) AS local_date, COUNT(*) AS hours
     FROM qa_missing_generation GROUP BY 1""")

In [ ]:
q("""SELECT CAST(period AS DATE) AS utc_date, COUNT(*) AS hours, ROUND(AVG(value)) AS avg_oth_mwh
     FROM qa_outliers WHERE issue LIKE 'OTH spike%' GROUP BY 1 ORDER BY 1""")

## Window functions (sql/04_windows.sql)

In [ ]:
q("SELECT * FROM peak_growth_yearly")

In [ ]:
q("SELECT year, local_ts, demand_mwh, ROUND(avg_temp_c,1) AS avg_temp_c FROM top_peaks_yearly WHERE peak_rank <= 3")

In [ ]:
# Spot check: lag and ramp columns vs raw data for three hours
q("""SELECT period, demand_mwh, demand_lag_24, demand_lag_168, ramp_1h, demand_7d_avg
     FROM model_hourly
     WHERE period IN ('2021-02-15 06:00+00', '2023-08-10 23:00+00', '2025-06-01 18:00+00')""")

In [ ]:
q("""SELECT period, demand_mwh FROM raw_demand
     WHERE period IN ('2023-08-10 22:00+00','2023-08-09 23:00+00','2023-08-03 23:00+00','2023-08-10 23:00+00')
     ORDER BY period""")

## Fuel mix and solar (sql/05_fuel_mix.sql)

In [ ]:
q("SELECT * FROM fuel_share_yearly")

In [ ]:
q("SELECT * FROM midday_solar_share")

## Net load and the duck curve (sql/06_net_load.sql)

In [ ]:
q("SELECT * FROM evening_ramp_yearly")

In [ ]:
q("SELECT * FROM top_evening_ramps WHERE ramp_rank <= 2")

## Winter Storm Uri (sql/07_uri.sql)

In [ ]:
q("SELECT * FROM uri_daily")

In [ ]:
u = q("SELECT * FROM uri_hourly")
peak = u.loc[u.demand_mwh.idxmax()]
after = u[(u.local_ts > peak.local_ts) & (u.local_date <= pd.Timestamp('2021-02-18'))]  # outage window Feb 15-18
trough = after.loc[after.total_gen_mwh.idxmin()]
uri = {
    "peak demand hour (local)": peak.local_ts,
    "peak served demand MWh": peak.demand_mwh,
    "total generation at peak MWh": peak.total_gen_mwh,
    "lowest generation during outages (local)": trough.local_ts,
    "lowest generation MWh": trough.total_gen_mwh,
    "generation drop MWh": peak.total_gen_mwh - trough.total_gen_mwh,
    "generation drop %": round(100 * (peak.total_gen_mwh - trough.total_gen_mwh) / peak.total_gen_mwh, 1),
    "gas at peak MWh": peak.gas_mwh, "gas at trough MWh": trough.gas_mwh,
    "coal at peak MWh": peak.coal_mwh, "coal at trough MWh": trough.coal_mwh,
    "nuclear at peak MWh": peak.nuclear_mwh, "nuclear at trough MWh": trough.nuclear_mwh,
    "wind at peak MWh": peak.wind_mwh, "wind at trough MWh": trough.wind_mwh,
    "temp at trough C": round(trough.avg_temp_c, 1),
    "storm-week wind vs baseline %": round(100 * (u.wind_mwh.mean() / u.base_wind.mean() - 1), 1),
    "missing hours in storm week": int(u.demand_mwh.isna().sum() + u.total_gen_mwh.isna().sum()),
}
outage = u[(u.local_date >= pd.Timestamp("2021-02-15")) & (u.local_date <= pd.Timestamp("2021-02-18"))]
uri.update({
    "ERCOT forecast peak during outages MWh": outage.ercot_forecast_mwh.max(),
    "max hourly forecast minus served MWh": outage.forecast_minus_served.max(),
    "hour of max gap (local)": outage.loc[outage.forecast_minus_served.idxmax(), "local_ts"],
    "forecast minus served, Feb 15-18 total GWh": round(outage.forecast_minus_served.sum() / 1000),
})
pd.Series(uri, dtype=object)

## Views and performance (sql/08_views.sql)

In [ ]:
for v in ["daily_peak", "fuel_mix_yearly", "net_load_hourly"]:
    display(q(f"SELECT * FROM {v} LIMIT 5"))

In [ ]:
import re
def explain_ms(sql):
    plan = con.execute("EXPLAIN ANALYZE " + sql).fetchall()[0][1]
    return float(re.search(r"Total Time: ([0-9.]+)s", plan).group(1)) * 1000

rolling = """SELECT period, AVG(demand_mwh) OVER (ORDER BY period ROWS BETWEEN 167 PRECEDING AND CURRENT ROW)
             FROM model_hourly"""
uri_self_join = open("sql/07_uri.sql").read().split("CREATE OR REPLACE TABLE uri_hourly AS")[1].split(";")[0]
timings = pd.DataFrame({
    "query": ["rolling 7-day average (61k rows)", "Uri baseline self-join"],
    "median_ms_of_5": [np.median([explain_ms(s) for _ in range(5)]) for s in (rolling, uri_self_join)],
}).round(1)
timings